## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)


In [23]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # this is the LLM abstraction

# from langchain_anthropic import ChatAnthropic # these LLM abstractions are swappable and can be called be invoke()

from langchain_chroma import Chroma  # this is the retriever abstraction (see notion)
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [24]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2


In [25]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)  # must MATCH THE EMBEDDING MODEL THAT YOU USE TO CONVERT THE DOCUMENTS TO VECTORS. because this is the model that we use to convert the question to vectors, if the embedding model between them don't match, trouble will ensue.
# we are creating a 300 something dimension vector from huggingface and trying to compare it with a 3000 dimension something vector with OpenAI, completely different vectors. we will get error
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":

- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right

- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!


In [26]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`


In [27]:
retriever.invoke(
    "Who is Avery?"
)  # this will retrieve relevant document objects that relevant to the question, 'Who is Avery?'

# Basically the retriever converted the question to vector using the OpenAI embedding model, then call Chroma Database to find the vector that most closely matches it

[Document(id='fbd9dbf1-0774-449e-9f47-107d37daff61', metadata={'doc_type': 'employees', 'source': '/Users/tayjiasheng/AI Projects/llm_engineering/week5/knowledge-base/employees/Avery Lancaster.md'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000'),
 Document(id='e16b163d-87e0-4f73-8ee7-a0a0900b3849', metadata={'doc_type': 'employees', 'source': '/Users/tayjiasheng/AI Projects/llm_engineering/week5/knowledge-base/employees/Avery Lancaster.md'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leade

In [28]:
llm.invoke("Who is Avery?")
# this is just a wrapper around openAI API, we are just getting back a generic answer from calling openai.chat.completions.create()
# it is not aware from the context aka the relevant documents from our retriever

AIMessage(content='Avery is a given name that can be used for both males and females. It may also refer to various people, characters, or entities depending on the context. Could you please provide more details or specify which Avery you are referring to?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 11, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7d0ce1a5ac', 'id': 'chatcmpl-DdzB8KB7r9IXlxsEhI05Y74zdrQEX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--ae8211f1-70f8-401c-824d-46433c572513-0', usage_metadata={'input_tokens': 11, 'output_tokens': 48, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cach

## Time to put this together!


In [29]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

# note that {context} is a placeholder here, NOT AN f-string!

In [30]:
# this entire function is just RAG.
def answer_question(question: str, history):
    # print(history)
    # we put the history params here for gradio! - without this gradio cannot display the chat history and we get error. gradio will auto pass in this arg so we don't have to use it in our function
    docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in docs
    )  # lay the stuff out and convert it to a string

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        context=context
    )  # substitute the placeholder in our system prompt

    response = llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=question)]
    )  # the HumanMessage is just the user message. instead of a list of dicts, it's just a list of objects
    return response.content

In [31]:
answer_question(
    "Who is Averi Lancaster?", []
)  # yes typos work too because the word is similar
# answer_question("What is the apex contract?", [])

'It seems like there might be a typo in the name. If you are referring to Avery Lancaster, she is the Co-Founder and CEO of Insurellm. She has been with the company since 2015 and is known for her leadership and expertise in the insurance technology industry. If you meant someone else, please let me know!'

## What could possibly come next? 😂


In [32]:
gr.ChatInterface(answer_question).launch()

# gradio is running the answer_question function!

/Users/tayjiasheng/AI Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!
